# 1 Этап. Аудит и подготовка корпуса

## 1. Загрузка

In [1]:
import json

with open("filmoskop_kb.json", encoding="utf-8") as f:
    data = json.load(f)
docs = data["docs"]        # 181 раздел базы знаний
queries = data["queries"]  # 66 тестовых запросов с разметкой

print("Разделов:", len(docs), "| Запросов:", len(queries))

Разделов: 181 | Запросов: 66


## 2. Поиск проблемных документов

In [2]:
texts = []
htmls = []
markdowns = []
for doc in docs:
    if doc["format"] == "text":
        texts.append(doc)
    elif doc["format"] == "html":
        htmls.append(doc)
    else:
        markdowns.append(doc)

In [3]:
for text in texts:
    print(text["doc_id"], " ", "*"*50)
    print(text["raw"])

0   **************************************************
Промокод даёт скидку на подписку или бесплатные дни. Ввести его можно в разделе «Подписка» — поле «У меня есть промокод» — или при оформлении подписки.

Правила простые: один промокод на один аккаунт, скидки по промокодам не суммируются между собой и с другими акциями, у каждого кода есть срок действия — он указан в условиях акции. Если код не применяется, проверьте срок действия и то, что он ещё не был использован на этом аккаунте.
1   **************************************************
Всё, что вы делаете в «Фильмоскопе», хранится не на устройстве, а в аккаунте. Поэтому телефон, планшет, браузер и телевизор показывают одно и то же: вошли — и сервис выглядит привычно, даже если устройство чужое.

Общее для аккаунта: подписка и тариф, способ оплаты, список привязанных устройств. Это не зависит ни от профиля, ни от того, откуда вы зашли.

Общее внутри профиля: список «Моё», оценки «Нравится» и «Не нравится», купленные фильмы и н

In [4]:
for html in htmls:
    print(html["doc_id"], " ", "*"*50)
    print(html["raw"])
    

8   **************************************************
<div class="kb-article"><h1>Пользовательское соглашение (выдержка): контент</h1>
<p>Пользовательское соглашение (выдержка). Раздел 9: контент 9.1. Все материалы сервиса охраняются авторским правом и предоставляются исключительно для личного просмотра. 9.2. Запрещается запись, копирование и распространение контента любыми средствами, включая запись экрана. 9.3. Состав каталога может изменяться в связи с условиями лицензионных соглашений с правообладателями.</p>
</div>
19   **************************************************
<div class="kb-article"><h1>«Фильмоскоп Бокс»: скоро</h1>
<p>«Фильмоскоп Бокс»: скоро Мы готовим собственную ТВ-приставку — «Фильмоскоп Бокс». Пульт с горячей кнопкой сервиса, быстрый запуск и фирменный интерфейс на большом экране. Технические характеристики и дату старта продаж объявим отдельно. Подпишитесь на новости в приложении, чтобы не пропустить анонс.</p>
</div>
23   ***************************************

In [5]:
for markdown in markdowns:
    print(markdown["doc_id"], " ", "*"*50)
    print(markdown["raw"])
    

2   **************************************************
# Безопасность аккаунта: общие советы

Несколько привычек, которые защищают аккаунт лучше любых настроек:

используйте уникальный пароль, который не повторяется на других сайтах; не вводите данные аккаунта на сторонних сайтах и не переходите по ссылкам из подозрительных писем «от сервиса»; не делитесь аккаунтом с людьми вне семьи; периодически просматривайте список устройств в настройках.

Помните: сервис никогда не запрашивает пароль в письмах и звонках.

6   **************************************************
# 10 фильмов для уютного вечера

10 фильмов для уютного вечера: выбор редакции

Собрали проверенные картины, которые согревают лучше пледа: тёплые комедии, добрые драмы и пара мюзиклов.

В подборке — истории о маленьких городках и больших надеждах, семейных ужинах и случайных встречах, которые меняют всё. Каждый фильм отобран редакцией вручную и доступен в отличном качестве.

Подборка ждёт вас на главной — полка «Уютный вечер

| № | Формат данных | Проблема / особенность | Комментарий |
|---|---------------|------------------------|-------------|
| 1 | `text` | Переносы строк и слов | Ломают структуру текста |
| 2 | `text` | Метаданные в документах | Не несут полезного смысла |
| 3 | `text` | «Краказябры» в русском языке | Проблемы с кодировкой |
| 4 | `text` | Неактуальные данные в БД | Могут давать неверный результат даже при идеальном поиске и модели |
| 5 | `text` | Технические детали | Например, `doc_id=75` — не помогает поиску |
| 6 | `text` | SQL‑запросы | Например, `doc_id=80` — бесполезны как база знаний |
| 7 | `text` | Спецсимволы (вероятно, из OCR) | Шум в данных |
| 8 | `text` | `doc_id=153` | Не несёт информации |
| 9 | `html` | Наличие тегов | Требуют очистки/парсинга |
|10 | `html` | Наличие дубликатов | Удалить дубликаты в тексте |
|11 | `html` | Метаданные в документах | Не несут полезного смысла |
|12 | `html` | `doc_id=156` | Не несёт информации |
|13 | `markdown` | «Краказябры» в русском языке | Проблемы с кодировкой |
|14 | `markdown` | Переносы строк и слов | Ломают структуру текста |
| — | — | **Всего разделов данных** | **181** |

## 4. Очистка 

In [ ]:
import re
import unicodedata
from bs4 import BeautifulSoup
JUNK_THRESHOLD = 0.15
MIN_LENGTH = 30


def junk_ratio(text: str) -> float:
    """
    Calculate junk ratio in text

    Args:
        text: input text
    
    Returns:
        junk ratio
    """
    if not text:
        return 0.0
    junk = re.findall(r"[^\w\s.,!?;:()«»—-]", text)
    return len(junk) / len(text)


def _cyr(s: str) -> float:
    """
    Сalculate cyrillic rate in text

    Args:
        s: input text
    
    Returns:
        cyrillic ratio
    """
    return sum(1 for c in s if '\u0400' <= c <= '\u04FF') / max(len(s), 1)


def _garbage(s: str) -> float:
    """
    Calculate junk ratio in text

    Args:
        s: input text
    
    Returns:
        junk ratio
    """
    return sum(
        1 for c in s
        if '\u0080' <= c <= '\u00ff'
        or unicodedata.category(c) in ('Cc', 'Cf', 'Cn', 'Co')
    ) / max(len(s), 1)


def fix_mojibake(s: str) -> str:
    """
    Search best score encoded text from mojibake to cyrillic

    Args:
        s: input text

    Returns:
        best text
    """
    if not isinstance(s, str):
        return s

    best = s

    best_score = (-_garbage(s), _cyr(s))

    for enc_from, enc_to in [
        ('latin-1', 'utf-8'),
        ('cp1251',  'utf-8'),
        ('latin-1', 'cp1251'),
        ('cp1251',  'latin-1'),
    ]:
        try:
            cand = s.encode(enc_from).decode(enc_to)
        except (UnicodeEncodeError, UnicodeDecodeError):
            continue

        score = (-_garbage(cand), _cyr(cand))
        if score > best_score:
            best, best_score = cand, score

    return best


def normalize_text(text: str) -> str:
    """
    Normalize text

    Args:
        text: input text

    Returns:
        normalized text
    """

    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    text = text.replace("\N{NO-BREAK SPACE}", " ")
    text = text.replace("\N{ZERO WIDTH SPACE}", " ")

    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"#", "", text)
    return text.strip()


def clear_text(raw) -> str:
    """
    Clear text

    Args:
        raw: input raw text

    Returns:
        cleared text
    """
    ratio = junk_ratio(raw)
    
    if ratio > JUNK_THRESHOLD:
        cleared_text = fix_mojibake(raw)
    else:
        cleared_text = raw
    
    cleared_text = normalize_text(cleared_text)
    return cleared_text


def prepare_html(raw: str) -> str:
    """
    Prepare html, delete tags

    Args:
        raw: input raw text

    Returns:
        cleared text
    """
    soup = BeautifulSoup(raw, "html.parser")
    for tag in soup(["script", "style", "nav", "footer"]):
        tag.decompose()
    for node in soup.select(".cookie"):
        node.decompose()
    
    return soup.get_text(" ")


def show_results(func):
    """
    Show results after prepare text

    Args:
        func: function for different types of text
    
    Returns:
        wrapped function
    """
    def wrapper(data):
        print(f"{data['doc_id']}: {data['format']}", "*"*50)
        raw = data["raw"]
        
        print("Было:\n", raw)
        print(f"Длина текста: {len(raw)}, кол-во слов: {len(raw.split())}")

        if data["format"] == "html":
            raw = prepare_html(raw)
        cleared_data = func(raw)
        
        print()
        print("Стало:\n", cleared_data)
        print(f"Длина: {len(cleared_data)}, кол-во слов: {len(cleared_data.split())}")

        return cleared_data
    
    return wrapper


@show_results
def decide(text) -> str:
    if len(text) == 0:
        return ""
    if len(text) < MIN_LENGTH:
        return ""
    if "Внимание: статья устарела и хранится в архиве." in text:
        return ""
    cleared_text = clear_text(text)
    return cleared_text.strip()

## 4. Оценка

In [7]:
for i, doc in enumerate(docs):
    docs[i]["cleared_text"] = decide(doc)

0: text **************************************************
Было:
 Промокод даёт скидку на подписку или бесплатные дни. Ввести его можно в разделе «Подписка» — поле «У меня есть промокод» — или при оформлении подписки.

Правила простые: один промокод на один аккаунт, скидки по промокодам не суммируются между собой и с другими акциями, у каждого кода есть срок действия — он указан в условиях акции. Если код не применяется, проверьте срок действия и то, что он ещё не был использован на этом аккаунте.
Длина текста: 441, кол-во слов: 73

Стало:
 Промокод даёт скидку на подписку или бесплатные дни. Ввести его можно в разделе «Подписка» — поле «У меня есть промокод» — или при оформлении подписки. Правила простые: один промокод на один аккаунт, скидки по промокодам не суммируются между собой и с другими акциями, у каждого кода есть срок действия — он указан в условиях акции. Если код не применяется, проверьте срок действия и то, что он ещё не был использован на этом аккаунте.
Длина: 435, 

У нас имелись 3 формата данных: `text`, `html` и `markdown`. Каждый формат был вручную проверен, и по результатам сделаны выводы:

- для каждого формата выявлены типовые ошибки (таблица с ошибками приведена выше);
- написаны функции по форматированию данных (кодировка, переносы строк, удаление невалидных данных);
- подобраны пороги фильтров;
- для информативности добавлены длины текстов и количество слов.

После исправления текста новый очищенный текст записан в текущий словарь с ключом `cleared_text`.

Также наблюдались разные варианты кодировок, поэтому я добавил дополнительный перебор возможных вариантов.

В качестве доли мусора решил взять **0.15** — выглядело оптимально: 95% текстов имели долю мусора меньше 15%, а больше 15% имели тексты с неправильной кодировкой, но это быстро исправлялось.

# Этап 2. Чанкинг и эмбеддинги

## 1. Стратегии чанкинга

Выберу 2 варианта чанкинга: sentence-based и recursive. Fixed-size с overlap скорее всего порежет посередине и не будет логического завершения предложения. Semantic будет избыточен и более дорогим по сравнению с выбранными вариантами.

In [8]:
from transformers import AutoTokenizer
from razdel import sentenize
from langchain_text_splitters import RecursiveCharacterTextSplitter
import statistics


In [9]:
e5_base_tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")

## 2. Построение эмбеддингов

### Sentence split

In [ ]:
def start_sentence_chunks(text: str, max_tokens: int = 512) -> list[str]:
    """
    Split text into sentences

    Args:
        text: text which need to split,
        max_tokens: max_tokens per chunk

    Returns:
         list of chunks
    """

    sentences = [s.text for s in sentenize(text)]
    
    chunks = []
    current = ["passage:"]
    current_tokens = len(e5_base_tokenizer(f"passage:", add_special_tokens=True)["input_ids"])

    for sent in sentences:
        n_tokens = len(e5_base_tokenizer(sent, add_special_tokens=True)["input_ids"])

        if current and current_tokens + n_tokens > max_tokens:
            chunks.append(" ".join(current[1:]))
            current = ["passage:"]
            current_tokens = len(e5_base_tokenizer(f"passage:", add_special_tokens=True)["input_ids"])
        current.append(sent)
        current_tokens += n_tokens
    if current:
        chunks.append(" ".join(current[1:]))
    return chunks


### Recursive split

In [ ]:
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=500, #немного запас по токенам для content'a модели e5
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""]
)

def start_recursive_chunks(text, max_tokens: int = 512) -> list[str]:
    """
    Recursive text splitting

    Args:
        text: text which need to split,
        max_tokens: max_tokens per chunk

    Returns:
         list of chunks
    """
    chunks = splitter.split_text(text)

    for i, chunk in enumerate(chunks):
        content = f"passage: {chunk}"

        n_tokens = len(e5_base_tokenizer(content, add_special_tokens=True)["input_ids"])

        if n_tokens > max_tokens:
            print(f"WARN chunk_text: {chunk},  chunk {i}: {n_tokens} tokens")
    
    return chunks


In [12]:
sentence_chunks = []
recursive_chunks = []
for doc in docs:
    text = doc["cleared_text"]
    sentence_chunks += start_sentence_chunks(text, 512)
    recursive_chunks += start_recursive_chunks(text, 512)

## 3. Ручной просмотр

In [13]:
len(sentence_chunks), len(recursive_chunks), len(docs)

(184, 185, 181)

Видно, что recursive чаще разбивал на чанки. Ниже посмотрим на статистику

In [ ]:
MIN_TOKENS = 15

def is_truncated(chunk: str) -> bool:
    """Возвращает True, если чанк не заканчивается завершающим знаком препинания, — значит, оборван."""
    return not chunk.rstrip().endswith((".", "!", "?", "…"))


def evaluate(name: str, chunks: str):
    """Считает метрики качества разбивки: распределение размеров, число коротких и оборванных чанков."""
    sizes = [len(e5_base_tokenizer(f"passage: {c}", add_special_tokens=True)['input_ids']) for c in chunks]
    short = sum(size < MIN_TOKENS for size in sizes)
    truncated = sum(is_truncated(c) for c in chunks)
    print(f"{name}:")
    print(f"  чанков: {len(chunks)}")
    print(f"  размеры, токенов: {sorted(sizes)}")
    print(f"  min / медиана / max: {min(sizes)} / "
          f"{statistics.median(sizes):.0f} / {max(sizes)}")
    print(f"  короче {MIN_TOKENS} токенов: {short}, доля: {round(short / len(sizes), 3)}")
    print(f"  обрезаны посреди предложения: {truncated}")

In [15]:
evaluate("sentence-based (512)", sentence_chunks)
print()
evaluate("recursive-based (500/50)", recursive_chunks)

sentence-based (512):
  чанков: 184
  размеры, токенов: [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 18, 24, 25, 26, 32, 39, 41, 49, 59, 65, 73, 86, 89, 89, 89, 89, 91, 93, 94, 95, 95, 95, 96, 97, 99, 100, 102, 102, 102, 102, 102, 102, 102, 103, 103, 104, 105, 105, 107, 108, 108, 109, 109, 109, 110, 110, 112, 112, 112, 115, 116, 117, 118, 119, 119, 120, 120, 120, 121, 121, 121, 122, 123, 124, 125, 125, 126, 126, 126, 128, 130, 130, 131, 131, 132, 133, 133, 134, 139, 145, 152, 157, 157, 160, 161, 162, 162, 168, 171, 171, 177, 181, 182, 188, 192, 197, 205, 205, 209, 220, 226, 226, 230, 232, 234, 241, 241, 243, 246, 247, 248, 252, 255, 259, 260, 263, 265, 266, 268, 269, 269, 271, 272, 272, 273, 275, 275, 277, 278, 278, 280, 280, 280, 280, 281, 283, 290, 292, 292, 293, 295, 298, 303, 316, 323, 330, 364, 384, 402, 406, 406, 413, 416, 427, 428, 433, 437, 448, 453, 455, 465, 471]
  min / медиана / max: 4 / 129 / 471
  короче 15 токенов: 22, доля: 0.12
  обрезаны посреди 

Видим, что **нет ни одного чанка, равного 512 токенам**, — отсюда можно сделать вывод, что все документы меньше данного значения.

- **Recursive-чанкер обрезает предложения чаще**, чем sentence — скорее всего, не все документы оканчиваются на специальный символ (`.!?...`).
- **Очень много документов меньше 100 токенов:**
  - доля в `sentence` = **18%**
  - доля в `recursive` = **18.5%**

Удалять такое количество документов не хотелось бы, поэтому нужно **уменьшить порог минимального количества токенов хотя бы до 30**:

| Чанкер | Доля документов < 30 токенов |
|---|---|
| `sentence` | 3.3% |
| `recursive` | 2% |

На основе вышесказанного я бы оставил **sentence-чанкер**, так как 1 документ помещается в 1 чанк и дробить не имеет смысла.

# Этап 3. Поисковая инфраструктура и retrieval-пайплайн

In [16]:
from sentence_transformers import SentenceTransformer

## 1. Хранилище

Возьмем Chroma, так как нам нужно работать локально и хранить метаданные

In [17]:
import chromadb
import torch
import numpy as np
from typing import Any

In [18]:
device = "cuda" if torch.cuda.is_available() else "cpu"
e5_base_embedder = SentenceTransformer("intfloat/multilingual-e5-base", device=device)

## 2. Настройка

In [ ]:
def embed_passages(model: SentenceTransformer, texts: str, batch_size: int = 64) -> np.ndarray:
    """
    Encode input text

    Args:
        model: embedder model
        texts: text which need to encode
        batch_size: batch size to encode text
    
    Returns:
        embed texts
    """
    inputs = ["passage: " + t for t in texts]
    return model.encode(inputs, normalize_embeddings=True, batch_size=batch_size)

all_chunks = []

for doc in docs:
    text = doc["cleared_text"]
    sentence_chunks = start_sentence_chunks(text, 512)
    
    for i, chunk in enumerate(sentence_chunks):
        if len(e5_base_tokenizer(f"passage: {chunk}", add_special_tokens=True)["input_ids"]) < MIN_TOKENS:
            continue
        all_chunks.append({
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "category": doc["category"],
            "chunk_index": i,
            "chunk_text": chunk
            })
        
texts = [t["chunk_text"] for t in all_chunks]
metas = [{k: v[k] for k in ("doc_id", "title", "category", "chunk_index")}
         for v in all_chunks]

embeddings = np.ascontiguousarray(embed_passages(e5_base_embedder, texts), dtype="float32")
print(f"Чанков: {len(texts)}, эмбеддинги: {embeddings.shape}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Чанков: 162, эмбеддинги: (162, 768)


In [20]:
client = chromadb.PersistentClient(path="local_save")
try:
    client.delete_collection("knowledge_base")
except Exception as e:
    print(e)

col = client.create_collection(
    name="knowledge_base",
    configuration={"hnsw": {"space": "cosine"}})


col.add(
    ids = [str(i) for i in range(len(texts))],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metas
)

## 3. Сборка retrieval пайплайна

In [21]:
def retrieve(query, col, embedder, k=5, threshold=0.0, filter: dict[str, Any] | None = None):
    """Retrieval-пайплайн: embed query -> search -> filter -> return."""
    q_vec = embedder.encode("query: " + query, normalize_embeddings=True)

    if filter:
        res = col.query(query_embeddings=[q_vec.tolist()], n_results=k, where=filter)
    else:
        res = col.query(query_embeddings=[q_vec.tolist()], n_results=k)
    out = []
    for cid, doc, dist in zip(res["metadatas"][0], res["documents"][0],
                              res["distances"][0]):
        sim = 1.0 - dist
        
        if sim >= threshold:
            out.append((cid["doc_id"], cid["title"], cid["category"], doc, sim))
    return out

In [22]:
for qid, query in enumerate(queries):
    rel_id = query['relevant_docs']
    hits = retrieve(query["query"], col, e5_base_embedder)

    print(f"\n[{qid}] {query['query']}")
    for cid, title, category, doc, sim in hits:
        mark = "+" if int(cid) in rel_id else " "
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:60]}, category={category}")



[0] что означает ошибка P-102
    + sim=0.862 #85 Ошибка P-102: превышен лимит просмотров Ошибка P-102 появляе, category=Ошибки воспроизведения
      sim=0.861 #178 Ошибка P-109 появляется, когда на устройстве не хватает своб, category=Ошибки воспроизведения
      sim=0.858 #137 Ошибка P-106 появляется, когда плееру не удаётся загрузить с, category=Ошибки воспроизведения
      sim=0.856 #138 Ошибка P-104 появляется, когда устройство не поддерживает за, category=Ошибки воспроизведения
      sim=0.856 #129 Ошибка P-105 появляется, когда на устройстве установлена уст, category=Ошибки воспроизведения

[1] какая версия webOS или Tizen нужна для приложения на телевизоре
    + sim=0.869 #146 На каких устройствах работает «Фильмоскоп» Телевизоры Smart , category=Устройства и качество
      sim=0.858 #86 Поддерживаемые устройства На каких устройствах работает "Фил, category=Устройства и качество
      sim=0.839 #139 Если вы не знаете, к какой платформе относится ваш телевизор, category=Устройс

Ручная проверка выявила, что оптимальным порогом будет 0.82, в такоем случае сломается только 36 запрос, так как релевантный запрос имеет сходство 0.806, в остальных же случаях релевантные ответы не пропадут, но при этом мы исключим шумные данные

## 4,5. Порог и фильтрация

In [23]:
for qid, query in enumerate(queries):
    rel_id = query['relevant_docs']
    hits = retrieve(query["query"], col, e5_base_embedder, threshold=0.82, filter={"category": 'Подписка и оплата'})

    print(f"\n[{qid}] {query['query']}")

    if not hits:
        print("    (ничего выше порога — пайплайн вернул пусто)")

    for cid, title, category, doc, sim in hits:
        mark = "+" if int(cid) in rel_id else " "
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:60]}, category={category}")


[0] что означает ошибка P-102
    (ничего выше порога — пайплайн вернул пусто)

[1] какая версия webOS или Tizen нужна для приложения на телевизоре
    (ничего выше порога — пайплайн вернул пусто)

[2] обязателен ли HDCP 2.2 для просмотра в 4K
    (ничего выше порога — пайплайн вернул пусто)

[3] как подключить саундбар по HDMI ARC
    (ничего выше порога — пайплайн вернул пусто)

[4] что означает ошибка P-109
    (ничего выше порога — пайплайн вернул пусто)

[5] что значит ошибка P-111
    (ничего выше порога — пайплайн вернул пусто)

[6] сколько стоит тариф Максимум и что в него входит
      sim=0.847 #24 Какой тариф подойдёт именно вам В «Фильмоскопе» три тарифа: , category=Подписка и оплата
    + sim=0.843 #113 Тарифы «Фильмоскопа» Тариф выбирается при оформлении подписк, category=Подписка и оплата
      sim=0.827 #32 Почему зрители выбирают «Максимум» Почему зрители выбирают «, category=Подписка и оплата
      sim=0.825 #29 Стоимость тарифов пересматривается редко, и о каждом изм

## 6. Baseline-метрики

In [24]:
def precision_at_k(found_ids, relevant_ids, k):
    """Доля релевантных среди k выданных чанков."""
    hits = [i for i in found_ids[:k] if i in relevant_ids]
    return len(hits) / k


def recall_at_k(found_ids, relevant_ids, k):
    """Доля релевантных чанков, попавших в top-k."""
    hits = [i for i in found_ids[:k] if i in relevant_ids]
    return len(hits) / len(relevant_ids)


def reciprocal_rank(found_ids, relevant_ids):
    """Обратный ранг первого релевантного чанка; 0, если его нет в выдаче."""
    for rank, i in enumerate(found_ids, start=1):
        if i in relevant_ids:
            return 1.0 / rank
    return 0.0

In [25]:
rows = []
queries_filter = []
for qid, query in enumerate(queries):

    rel_id = query['relevant_docs']
    if not len(rel_id):
        continue
    hits = retrieve(query["query"], col, e5_base_embedder, k=5)
    found_doc_id = [hit[0] for hit in hits]
    categories = set([hit[2] for hit in hits])
    queries_filter.append(query)
    rows.append({
        "n_rel": len(rel_id),
        "precision": precision_at_k(list(set(found_doc_id)), rel_id, 5),
        "recall": recall_at_k(list(set(found_doc_id)), rel_id, 5),
        "rr": reciprocal_rank(found_doc_id, rel_id),
        "categories": categories
    })

In [26]:
print(f"{'индекс':<10}{'запрос':<44}{'|R|':>4}{'P@5':>7}{'R@5':>7}{'RR':>7}{'CATS':>7}")

for i, (row, q) in enumerate(zip(rows, queries_filter)):
    print(f"{i:<10}{q['query'][:42]:<44}{row['n_rel']:>4}"
          f"{row['precision']:>7.2f}{row['recall']:>7.2f}{row['rr']:>7.2f}   {row['categories']}")
print(f"\nprecision@5 = {np.mean([r['precision'] for r in rows]):.3f}")
print(f"recall@5    = {np.mean([r['recall'] for r in rows]):.3f}")
print(f"MRR         = {np.mean([r['rr'] for r in rows]):.3f}") 

индекс    запрос                                       |R|    P@5    R@5     RR   CATS
0         что означает ошибка P-102                      1   0.20   1.00   1.00   {'Ошибки воспроизведения'}
1         какая версия webOS или Tizen нужна для при     1   0.20   1.00   1.00   {'Ошибки воспроизведения', 'Устройства и качество'}
2         обязателен ли HDCP 2.2 для просмотра в 4K      1   0.20   1.00   1.00   {'Ошибки воспроизведения', 'Устройства и качество'}
3         как подключить саундбар по HDMI ARC            1   0.00   0.00   0.00   {'Ошибки воспроизведения', 'Устройства и качество'}
4         что означает ошибка P-109                      1   0.20   1.00   1.00   {'Ошибки воспроизведения'}
5         что значит ошибка P-111                        1   0.20   1.00   1.00   {'Ошибки воспроизведения'}
6         сколько стоит тариф Максимум и что в него      1   0.20   1.00   0.50   {'Ошибки воспроизведения', 'Подписка и оплата'}
7         сколько дней длится пробный период          

In [27]:
rows = {}
top_k_list = [1, 3, 5, 10, 20, 30]
for qid, query in enumerate(queries):

    rel_id = query['relevant_docs']
    if not len(rel_id):
        continue
    rows[qid] = []
    for top_k in top_k_list:
        hits = retrieve(
            query=query["query"],
            col=col,
            embedder=e5_base_embedder,
            k=top_k
            )
        found_doc_id = [hit[0] for hit in hits]
        categories = set([hit[2] for hit in hits])

        rows[qid].append(recall_at_k(list(set(found_doc_id)), rel_id, top_k))

In [28]:
n_rows = len(next(iter(rows.values())))   # 5

recall_at_topk = [
    sum(rows[col][i] for col in rows) / len(rows)
    for i in range(n_rows)
]

print(f"{'Значение k':>4}", f"{'recall':<10}")
for top, recall in zip(top_k_list, recall_at_topk):
    print(f"{top:>10}", f"{round(recall, 4):<10}")

Значение k recall    
         1 0.4196    
         3 0.6161    
         5 0.7232    
        10 0.8929    
        20 0.9821    
        30 1.0       


Видим, что при recall@30=1.0. Для реранкера можем брать этот порог

# Этап 4. Диагностика и улучшение retrieval

Определим проблемные запросы

### 1. Запросы с низкими метриками

In [29]:
problems_top = []
problems_general = []
all_hits = []
for qi, query in enumerate(queries_filter):
    hits = retrieve(query["query"], col, e5_base_embedder)
    all_hits.append(hits)
    top_ids = [int(hit[0]) for hit in hits]
    rel_id = query['relevant_docs']
    is_problem = (not top_ids) or (top_ids[0] not in rel_id)
    if is_problem:
        problems_top.append(qi)
    
    is_problem = (not top_ids) or (not set(top_ids).intersection(set(rel_id)))
    if is_problem:
        problems_general.append(qi)

print("Запросы не топ-1:", problems_top)
print("Запросы с невалидными ответами:", problems_general)

Запросы не топ-1: [3, 6, 10, 13, 15, 18, 19, 20, 22, 24, 28, 30, 31, 33, 34, 35, 37, 38, 39, 40, 41, 44, 46, 47, 48, 49, 51, 53, 55]
Запросы с невалидными ответами: [3, 18, 20, 22, 30, 33, 34, 35, 37, 39, 53]


### 2. Диагноз

Запросы, в которых документы находились в топ-5, но не топ-1 [3, 6, 10, 13, 15, 18, 19, 20, 22, 24, 28, 30, 31, 33, 34, 35, 37, 38, 39, 40, 41, 44, 46, 47, 48, 49, 51, 53, 55] <br>
Запросы и найденные документы сильно отличаются [3, 18, 20, 22, 30, 33, 34, 35, 37, 39, 53] <br>

In [30]:
for qid, query in enumerate(queries_filter):
    if qid not in problems_top:
        continue

    print(f"\n[{qid}] {query['query']}")
    for cid, title, category, doc, sim in all_hits[qid]:
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:80]}")



[3] как подключить саундбар по HDMI ARC
      sim=0.842 #125 Если при воспроизведении звук идет, а изображения нет — экран остается черным, —
      sim=0.832 #69 Объёмный звук дома собирают двумя способами, и выбор между ними — это выбор межд
      sim=0.825 #75 {"service": "player-api", "version": "2.14.3", "env": "production", "features": 
      sim=0.816 #50 Встроенные динамики телевизора делают тонкими вместе с корпусом, поэтому внешнюю
      sim=0.813 #71 Статья обновляется Раздел временно недоступен Статья обновляется. Загляните позж

[6] сколько стоит тариф Максимум и что в него входит
      sim=0.847 #24 Какой тариф подойдёт именно вам В «Фильмоскопе» три тарифа: «Мобильный», «Станда
      sim=0.843 #113 Тарифы «Фильмоскопа» Тариф выбирается при оформлении подписки и в любой момент в
      sim=0.828 #71 Статья обновляется Раздел временно недоступен Статья обновляется. Загляните позж
      sim=0.827 #32 Почему зрители выбирают «Максимум» Почему зрители выбирают «Максимум» «Макс

3 запрос имеет термин HDMI ARC, релевантный документ doc_id=22, но он не был извлечен вообще в топ-5; <br>
47 запрос имеет точный термин P-103, документ нашелся, но не топ-1; <br>
очень часто в топ попадает 71 документ (Статья обновляется - будто бы документ вообще не несет полезной информации), 39 документ (видимо это просто наименования столбцов), 75 документ (просто json файл), 80 документ (sql запрос)


### 3. Методы улучшения

#### Reranker

In [31]:
from sentence_transformers import CrossEncoder

In [32]:
device = "cuda" if torch.cuda.is_available() else "cpu"
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device)

In [33]:
def rerank(query: str, hits: list[tuple[Any]], reranker: CrossEncoder, k: int = 5):
    pairs = [(query, hit[3]) for hit in hits]
    scores = reranker.predict(pairs)
    order = [hits[i] for i in np.argsort(scores)[::-1][:k]]

    return order

In [34]:
all_reranked = []
for qid, query in enumerate(queries_filter):
    rel_id = query['relevant_docs']
    hits = retrieve(query["query"], col, e5_base_embedder, k=30)
    
    reranked = rerank(query["query"], hits, reranker)
    all_reranked.append(reranked)
    print(f"\n[{qid}] {query['query']}")
    print(f"  Биэнкодер top-5 (до реранкинга):")
    for cid, title, category, doc, sim in hits[:5]:
        mark = "+" if int(cid) in rel_id else " "
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:60]}, category={category}")
    print(f"  После реранкинга top-5:")
    for cid, title, category, doc, sim in reranked:
        mark = "+" if int(cid) in rel_id else " "
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:60]}, category={category}")


[0] что означает ошибка P-102
  Биэнкодер top-5 (до реранкинга):
    + sim=0.862 #85 Ошибка P-102: превышен лимит просмотров Ошибка P-102 появляе, category=Ошибки воспроизведения
      sim=0.861 #178 Ошибка P-109 появляется, когда на устройстве не хватает своб, category=Ошибки воспроизведения
      sim=0.858 #137 Ошибка P-106 появляется, когда плееру не удаётся загрузить с, category=Ошибки воспроизведения
      sim=0.856 #138 Ошибка P-104 появляется, когда устройство не поддерживает за, category=Ошибки воспроизведения
      sim=0.856 #129 Ошибка P-105 появляется, когда на устройстве установлена уст, category=Ошибки воспроизведения
  После реранкинга top-5:
    + sim=0.862 #85 Ошибка P-102: превышен лимит просмотров Ошибка P-102 появляе, category=Ошибки воспроизведения
      sim=0.848 #165 Ошибка P-112 появляется, когда аккаунт заблокирован за наруш, category=Ошибки воспроизведения
      sim=0.850 #90 Ошибка P-110 появляется, когда происходит сбой на стороне се, category=Ошибки воспрои

In [35]:
with open("all_reranked.json", "w") as f:
    json.dump(all_reranked, f, ensure_ascii=False)

In [36]:
rows = []

for qid, query in enumerate(queries_filter):
    rel_id = query['relevant_docs']
    all_reranked[qid]
    found_doc_id = [hit[0] for hit in all_reranked[qid]]
    categories = set([hit[2] for hit in all_reranked[qid]])

    rows.append({
        "n_rel": len(rel_id),
        "precision": precision_at_k(list(set(found_doc_id)), rel_id, 5),
        "recall": recall_at_k(list(set(found_doc_id)), rel_id, 5),
        "rr": reciprocal_rank(found_doc_id, rel_id),
        "categories": categories
    })

print(f"{'индекс':<10}{'запрос':<44}{'|R|':>4}{'P@5':>7}{'R@5':>7}{'RR':>7}{'CATS':>7}")

for i, (row, q) in enumerate(zip(rows, queries_filter)):
    print(f"{i:<10}{q['query'][:42]:<44}{row['n_rel']:>4}"
          f"{row['precision']:>7.2f}{row['recall']:>7.2f}{row['rr']:>7.2f}   {row['categories']}")
print(f"\nprecision@5 = {np.mean([r['precision'] for r in rows]):.3f}")
print(f"recall@5    = {np.mean([r['recall'] for r in rows]):.3f}")
print(f"MRR         = {np.mean([r['rr'] for r in rows]):.3f}") 


индекс    запрос                                       |R|    P@5    R@5     RR   CATS
0         что означает ошибка P-102                      1   0.20   1.00   1.00   {'Ошибки воспроизведения'}
1         какая версия webOS или Tizen нужна для при     1   0.20   1.00   1.00   {'Устройства и качество'}
2         обязателен ли HDCP 2.2 для просмотра в 4K      1   0.20   1.00   1.00   {'Подписка и оплата', 'Устройства и качество'}
3         как подключить саундбар по HDMI ARC            1   0.20   1.00   1.00   {'Ошибки воспроизведения', 'Устройства и качество'}
4         что означает ошибка P-109                      1   0.20   1.00   1.00   {'Ошибки воспроизведения'}
5         что значит ошибка P-111                        1   0.20   1.00   1.00   {'Ошибки воспроизведения'}
6         сколько стоит тариф Максимум и что в него      1   0.20   1.00   1.00   {'Функции и контент', 'Подписка и оплата', 'Устройства и качество'}
7         сколько дней длится пробный период             1   0.20

| Метрика      | До    | После |
|--------------|-------|-------|
| precision@5  | 0.189 | 0.250 |
| recall@5     | 0.723 | 0.955 |
| MRR          | 0.614 | 0.829 |

После добавления реранкера все метрики стали выше (precision@5 достигло почти своего предела, так как в целом релевантных документов в каждом запросе 1-2 шт.). Но за это качество мы платим временем. На CPU 56 запросов реранкер отработал за 15 минут (на вход подавали топ 30 документов, найденных на этапе BI-encoder), что очень много, нужна GPU для ускорения процесса.

#### Query expansion

In [ ]:
from openai import OpenAI
from collections import defaultdict

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
LLM = "qwen3.5:9b"
NO_THINK = {"reasoning_effort": "none"}

def generate(messages: list[dict[str, str]], max_tokens: int = 350) -> str:
    """
    Answer generate by model

    Args:
        messages: history message to model input
        max_tokens: max_tokens generate by model

    Returns:
        answer
    """
    response = client.chat.completions.create(
            model=LLM,
            messages=messages,
            temperature=0.0,
            max_tokens=max_tokens,
            extra_body=NO_THINK)

    return response.choices[0].message.content.strip()


def expand(query, n=5):
    """Query expansion: n перефразировок запроса официальными словами базы."""
    text = generate([
        {"role": "system", "content": "Ты умный ассистент по базе знаний онлайн-кинотеатра."},
        {"role": "user", "content": f"Переформулируй запрос клиента {n} способами "
         f"официальными словами из базы знаний. Используй полные фразы (не одно "
         f"слово), синонимы и термины. По одной формулировке на строку, без "
         f"нумерации.\n\nЗапрос: {query}"}])
    return [l.strip() for l in text.splitlines() if l.strip()][:n]


def rrf(ranked_lists, k=60, top_k=5):
    """Reciprocal Rank Fusion: сливаем несколько выдач по рангам."""
    scores = defaultdict(float)
    for lst in ranked_lists:
        for rank, doc in enumerate(lst, start=1):
            scores[(doc[:-1])] += 1.0 / (k + rank)
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    return [doc[0] + (doc[1], ) for doc in ranked[:top_k]]

In [ ]:
def calculate_metrics(queries_filter: list[dict[str, Any]], calc_idx: list[int], hits: list[tuple[Any]]) -> tuple[float]:
    """
    Calculate precision, recall and mrr metrics

    Args:
        queries_filter: queries without gap kind
        calc_idx: queries_filter idx which need to calculate
        hits: found docs from retrieval
    
    Returns:
        tuple of metrics
    """
    rows = []

    for i, qid in enumerate(calc_idx):
        query = queries_filter[qid]
        rel_id = query['relevant_docs']

        found_doc_id = [hit[0] for hit in hits[i]]
        categories = set([hit[2] for hit in hits[i]])

        rows.append({
            "n_rel": len(rel_id),
            "precision": precision_at_k(list(set(found_doc_id)), rel_id, 5),
            "recall": recall_at_k(list(set(found_doc_id)), rel_id, 5),
            "rr": reciprocal_rank(found_doc_id, rel_id),
            "categories": categories
        })

    print(f"{'индекс':<10}{'запрос':<44}{'|R|':>4}{'P@5':>7}{'R@5':>7}{'RR':>7}{'CATS':>7}")

    for i, (row, q) in enumerate(zip(rows, queries_filter)):
        print(f"{i:<10}{q['query'][:42]:<44}{row['n_rel']:>4}"
            f"{row['precision']:>7.2f}{row['recall']:>7.2f}{row['rr']:>7.2f}   {row['categories']}")
    print(f"\nprecision@5 = {np.mean([r['precision'] for r in rows]):.3f}")
    print(f"recall@5    = {np.mean([r['recall'] for r in rows]):.3f}")
    print(f"MRR         = {np.mean([r['rr'] for r in rows]):.3f}")

    return np.mean([r['precision'] for r in rows]), np.mean([r['recall'] for r in rows]), np.mean([r['rr'] for r in rows])


In [39]:
raw_hits = []
reranked_without_exp_hits = []
reranked_with_hits = []
query_expansion_hits = []

for qid in problems_general:
    query = queries_filter[qid]
    rel_id = query['relevant_docs']

    hits = retrieve(query["query"], col, e5_base_embedder, k=30)
    raw_hits.append(hits)

    reranked_without = rerank(query["query"], hits, reranker)
    reranked_without_exp_hits.append(reranked_without)
    
    query_expansion = expand(query["query"])
    expanded = rrf([retrieve(query, col, e5_base_embedder, k=30)
                for query in query_expansion], top_k=30)
    query_expansion_hits.append(expanded)

    reranked = rerank(query["query"], expanded, reranker)
    reranked_with_hits.append(reranked)
    print(f"\n[{qid}] {query['query']}")
    
    print(f"  Биэнкодер top-5 (до реранкинга):")
    for cid, title, category, doc, sim in hits[:10]:
        mark = "+" if int(cid) in rel_id else " "
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:60]}, category={category}")
    
    print(f"  После реранкинга биэнкодера top-5:")
    for cid, title, category, doc, sim in reranked_without:
        mark = "+" if int(cid) in rel_id else " "
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:60]}, category={category}")

    print("  query expansion top5:")
    for cid, title, category, doc, sim in expanded[:10]:
        mark = "+" if int(cid) in rel_id else " "
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:60]}, category={category}")

    print(f"  После реранкинга query expansion top-5:")
    for cid, title, category, doc, sim in reranked:
        mark = "+" if int(cid) in rel_id else " "
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:60]}, category={category}")


[3] как подключить саундбар по HDMI ARC
  Биэнкодер top-5 (до реранкинга):
      sim=0.842 #125 Если при воспроизведении звук идет, а изображения нет — экра, category=Ошибки воспроизведения
      sim=0.832 #69 Объёмный звук дома собирают двумя способами, и выбор между н, category=Устройства и качество
      sim=0.825 #75 {"service": "player-api", "version": "2.14.3", "env": "produ, category=Ошибки воспроизведения
      sim=0.816 #50 Встроенные динамики телевизора делают тонкими вместе с корпу, category=Устройства и качество
      sim=0.813 #71 Статья обновляется Раздел временно недоступен Статья обновля, category=Ошибки воспроизведения
      sim=0.811 #150 Игры и кино на одной консоли У вас уже есть всё для домашнег, category=Устройства и качество
    + sim=0.811 #22 Обозначение 5.1 описывает раскладку каналов: пять основных к, category=Устройства и качество
      sim=0.811 #52 Раздел справки (архив) Страница не найдена Возможно, раздел , category=Устройства и качество
      sim=0.809

In [68]:
print("raw_hits: top5")
calculate_metrics(queries_filter, problems_general, raw_hits)

print("*"*50)
print("reranker expansion: top5")
calculate_metrics(queries_filter, problems_general, reranked_without_exp_hits)

print("*"*50)
print("query expansion: top5")
calculate_metrics(queries_filter, problems_general, query_expansion_hits)

print("*"*50)
print("reranker expansion: top5")
calculate_metrics(queries_filter, problems_general, reranked_with_hits)

raw_hits: top5
индекс    запрос                                       |R|    P@5    R@5     RR   CATS
0         что означает ошибка P-102                      1   0.00   0.00   0.14   {'Ошибки воспроизведения', 'Функции и контент', 'Аккаунт и профили', 'Устройства и качество'}
1         какая версия webOS или Tizen нужна для при     2   0.00   0.00   0.17   {'Ошибки воспроизведения', 'Аккаунт и профили', 'Подписка и оплата', 'Функции и контент', 'Устройства и качество'}
2         обязателен ли HDCP 2.2 для просмотра в 4K      1   0.00   0.00   0.17   {'Ошибки воспроизведения', 'Аккаунт и профили', 'Подписка и оплата', 'Устройства и качество'}
3         как подключить саундбар по HDMI ARC            1   0.00   0.00   0.06   {'Ошибки воспроизведения', 'Аккаунт и профили', 'Подписка и оплата', 'Функции и контент', 'Устройства и качество'}
4         что означает ошибка P-109                      1   0.20   1.00   0.07   {'Ошибки воспроизведения', 'Аккаунт и профили', 'Подписка и оплата', '

(np.float64(0.18181818181818182),
 np.float64(0.8181818181818182),
 np.float64(0.5454545454545454))

query_expansion дает более высокий recall и mrr по сравнению с обычным запросом, precision остался прежним (в целом у нас в среднем 1 документ релевантный на запрос), а вот реранкер по expansion показал хуже метрики чем на обычном поиске (но если взять уже не топ30 поиска а например всего лишь 10, то реранкер покажет себя лучше), так как я все равно в реранкер expansion передавал запрос обычный, но найденные документы были уже другими.

#### Гибридный поиск

In [41]:
import re
from collections import defaultdict
from rank_bm25 import BM25Okapi

In [ ]:
def tokenize(text):
    """Разбивает текст на токены-слова в нижнем регистре — вход для BM25."""
    return re.findall(r"\w+", text.lower())


def sparse_search(query, k=3):
    """Разреженный поиск BM25: top-k по совпадению слов с весом по редкости."""
    results = []
    
    scores = bm25.get_scores(tokenize(query))
    order = np.argsort(scores)[::-1][:k]

    for i in order:
        results.append((int(i), docs[i]['title'], docs[i]['category'], docs[i]['cleared_text'], float(scores[i])))
    return results
    #return [int(i) for i in order], list(float(score) for score in scores[order])


bm25 = BM25Okapi([tokenize(doc['cleared_text']) for doc in docs])

def rrf_fuse(ranked_lists, k=60, top_k=5):
    """Reciprocal Rank Fusion: сливаем несколько выдач по рангам."""
    scores = defaultdict(float)
    for lst in ranked_lists:
        for rank, doc in enumerate(lst, start=1):
            scores[(doc[:-1])] += 1.0 / (k + rank)

    ranked = sorted(scores.items(), key=lambda x: -x[1])
    return [doc[0] + (doc[1], ) for doc in ranked[:top_k]]


def hybrid_search(semantic, bm25, k=3):
    """Гибридный поиск: сливаем плотную и разреженную выдачи через RRF."""
    fused = rrf_fuse([semantic + bm25], k=60, top_k=k)
    return fused[:k]

In [43]:
all_hits = []
all_bm25_search = []
all_hybrid_search = []

for qid in problems_general:#range(len(queries_filter)):#problems_general:
#for qid in range(len(queries_filter)):#problems_general:
    query = queries_filter[qid]
    rel_id = query['relevant_docs']
    
    hits = retrieve(query["query"], col, e5_base_embedder, k=30)
    bm25_search = sparse_search(query["query"], k=30)
    hybrid = hybrid_search(hits, bm25_search, k=30)

    all_hits.append(hits)
    all_bm25_search.append(bm25_search)
    all_hybrid_search.append(hybrid)

    print(f"\n[{qid}] {query['query']}")
    
    print(f"  Retrieval top5:")
    for cid, title, category, doc, sim in hits[:10]:
        mark = "+" if int(cid) in rel_id else " "
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:60]}, category={category}")
    
    print(f"  BM25 top5:")
    for cid, title, category, doc, sim in bm25_search[:10]:
        mark = "+" if int(cid) in rel_id else " "
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:60]}, category={category}")

    print("  Hybrid top5:")
    for cid, title, category, doc, sim in hybrid[:30]:
        mark = "+" if int(cid) in rel_id else " "
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:60]}, category={category}")


[3] как подключить саундбар по HDMI ARC
  Retrieval top5:
      sim=0.842 #125 Если при воспроизведении звук идет, а изображения нет — экра, category=Ошибки воспроизведения
      sim=0.832 #69 Объёмный звук дома собирают двумя способами, и выбор между н, category=Устройства и качество
      sim=0.825 #75 {"service": "player-api", "version": "2.14.3", "env": "produ, category=Ошибки воспроизведения
      sim=0.816 #50 Встроенные динамики телевизора делают тонкими вместе с корпу, category=Устройства и качество
      sim=0.813 #71 Статья обновляется Раздел временно недоступен Статья обновля, category=Ошибки воспроизведения
      sim=0.811 #150 Игры и кино на одной консоли У вас уже есть всё для домашнег, category=Устройства и качество
    + sim=0.811 #22 Обозначение 5.1 описывает раскладку каналов: пять основных к, category=Устройства и качество
      sim=0.811 #52 Раздел справки (архив) Страница не найдена Возможно, раздел , category=Устройства и качество
      sim=0.809 #112 Если изобра

In [44]:
print("Retrieval: top30")
calculate_metrics(queries_filter, problems_general, all_hits)
#calculate_metrics(queries_filter, [qid for qid in range(len(queries_filter))], all_hits)

print("*"*50)
print("BM25: top30")
calculate_metrics(queries_filter, problems_general, all_bm25_search)#[qid for qid in range(len(queries_filter))], all_bm25_search)#problems_general, all_bm25_search)
#calculate_metrics(queries_filter, [qid for qid in range(len(queries_filter))], all_bm25_search)

print("*"*50)
print("Hybrid: top30")
calculate_metrics(queries_filter, problems_general, all_hybrid_search)#[qid for qid in range(len(queries_filter))], all_hybrid_search)#problems_general, all_hybrid_search)
#calculate_metrics(queries_filter, [qid for qid in range(len(queries_filter))], all_hybrid_search)

Retrieval: top30
индекс    запрос                                       |R|    P@5    R@5     RR   CATS
0         что означает ошибка P-102                      1   0.00   0.00   0.14   {'Ошибки воспроизведения', 'Функции и контент', 'Аккаунт и профили', 'Устройства и качество'}
1         какая версия webOS или Tizen нужна для при     2   0.00   0.00   0.17   {'Ошибки воспроизведения', 'Аккаунт и профили', 'Подписка и оплата', 'Функции и контент', 'Устройства и качество'}
2         обязателен ли HDCP 2.2 для просмотра в 4K      1   0.00   0.00   0.17   {'Ошибки воспроизведения', 'Аккаунт и профили', 'Подписка и оплата', 'Устройства и качество'}
3         как подключить саундбар по HDMI ARC            1   0.00   0.00   0.06   {'Ошибки воспроизведения', 'Аккаунт и профили', 'Подписка и оплата', 'Функции и контент', 'Устройства и качество'}
4         что означает ошибка P-109                      1   0.20   1.00   0.07   {'Ошибки воспроизведения', 'Аккаунт и профили', 'Подписка и оплата',

Precision и Recall у нас получился одинаковым так как мы берем топ30 и отсюда они одинаковые, однако видим, что у гибридного поиска MRR стал больше - это значит, что релевантные документы поднимаются наверх.

## 4. Сравнение

Очень хорошо показал себя реранкер, метрки очень хорошо выросли. В качестве дальнейших действий я бы брал гибридный поиск и реранкер.

## 5. Оценка

Цена улучшения это:
- время задержки увеличиватся с 0.5 секунд до 9 секунд за 1 запрос (все считалось на CPU);
- необходимо добавлять модель реранкера;
- необходимо задействовать GPU для ускорения расчета реранкером;
- добавляются новые функции по гибридному поиску и реранкеру 

# Этап 5. Генерация ответов

Применим гибридный поиск с реранкером

In [45]:
SYSTEM_PROMPT = (
    "Ты - умный ассистент по базе знаний онлайн-кинотеатра. "
    "Отвечай ТОЛЬКО по приведенным фрагментам базы знаний."
    "Если ответа в них нет, то ответь: «В базе знаний ответа нет»"
    "Отвечай кратко, одним-двумя предложениями."
)


def build_messages(query, found_context):
    """Собирает промпт: system-инструкция + нумерованный контекст + вопрос."""
    context = "\n".join(f"[{n}] {retrieval[3]}" for n, retrieval in enumerate(found_context, 1))
    user = f"Фрагменты базы знаний:\n{context}\n\nВопрос: {query}"
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user}]

In [371]:
for i, query in enumerate(queries):
    rel_id = query['relevant_docs']
    
    hits = retrieve(query["query"], col, e5_base_embedder, k=30)
    bm25_search = sparse_search(query["query"], k=30)
    hybrid = hybrid_search(hits, bm25_search, k=30)
    reranked_hits = rerank(query["query"], hybrid[:15], reranker)
    
    answer = generate(build_messages(query["query"], reranked_hits))

    queries[i]["predict_answer"] = answer
    print("="*70)
    print(f"\nВ: {query['query']}; type={query['kind']}")
    print(f"   чанки:")

    for cid, title, category, doc, sim in reranked_hits:
        mark = "+" if int(cid) in rel_id else " "
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:60]}, category={category}")
    print(f"   О: {answer}")

    


В: что означает ошибка P-102; type=exact
   чанки:
    + sim=0.027 #85 Ошибка P-102: превышен лимит просмотров Ошибка P-102 появляе, category=Ошибки воспроизведения
      sim=0.024 #165 Ошибка P-112 появляется, когда аккаунт заблокирован за наруш, category=Ошибки воспроизведения
      sim=0.025 #90 Ошибка P-110 появляется, когда происходит сбой на стороне се, category=Ошибки воспроизведения
      sim=0.025 #94 Ошибка P-101 появляется, когда приложению не удаётся соедини, category=Ошибки воспроизведения
      sim=0.024 #102 Если просмотр прерывается, плеер показывает короткий код вид, category=Ошибки воспроизведения
   О: Ошибка P-102 появляется, когда число одновременных просмотров на аккаунте превышает лимит тарифа. Для решения проблемы необходимо остановить просмотр на другом устройстве или перейти на тариф с большим числом одновременных просмотров.

В: какая версия webOS или Tizen нужна для приложения на телевизоре; type=exact
   чанки:
    + sim=0.027 #146 На каких устройствах раб

In [373]:
with open("queries_with_answers.json", "w") as f:
    json.dump(queries, f, ensure_ascii=False)

# Этап 6. Оценка качества и финальная диагностика

## 1. LLM judge

In [46]:
JUDGE_SYSTEM = (
    "Ты — строгий эксперт по оценке ответов справочных систем. Оценивай "
    "беспристрастно, длина ответа не является достоинством. "
    "Отвечай только JSON-объектом."
)

FAITH_PROMPT = """Оцени ВЕРНОСТЬ КОНТЕКСТУ (faithfulness):
следует ли каждое утверждение ответа из фрагментов.
Полезность ответа и его соответствие вопросу не оценивай —
только происхождение фактов.

Шкала:
5 — все утверждения следуют из фрагментов;
4 — все утверждения следуют из фрагментов,
добавлено безобидное обобщение;
3 — главное утверждение из фрагментов,
но добавлена деталь, которой в них нет;
2 — часть утверждений выдумана или противоречит фрагментам;
1 — ключевые факты ответа отсутствуют во фрагментах или искажены.
Ответ «В базе знаний ответа нет» не утверждает ничего и получает 5.

Фрагменты:
{context}

Ответ: {answer}

Верни JSON с двумя полями:
{{"reason": "<одно предложение>", "score": <целое число 1-5>}}"""


REL_PROMPT = """Оцени РЕЛЕВАНТНОСТЬ (answer relevance):
отвечает ли ответ на заданный вопрос по существу.
Правдивость не проверяй — только соответствие вопросу.

Шкала:
5 — прямой и полный ответ;
4 — ответ по существу с небольшими отступлениями;
3 — закрыта только часть вопроса;
2 — текст на нужную тему, но ответа нет;
1 — другая тема или отказ отвечать.

Вопрос: {question}

Ответ: {answer}

Верни JSON с полями reason и score."""


JUDGE_RESPONSE_FORMAT = {
    "type": "json_object",
    "schema": {
        "type": "object",
        "properties": {
            "reason": {"type": "string", "maxLength": 240},
            "score": {
                "type": "integer",
                "minimum": 1,
                "maximum": 5,
            },
        },
        "required": ["reason", "score"],
        "additionalProperties": False,
    },
}

In [ ]:
from openai import OpenAI
from collections import defaultdict

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
LLM = "qwen3.5:9b"
NO_THINK = {"reasoning_effort": "none"}


def generate_judge(messages: list[dict[str, str]], max_tokens: int = 350) -> str:
    """
    Generate response by judge llm

    Args:
        messages: hitstory messages
        max_tokens: max_tokens which model must generate answer
    
    Returns:
        answer
    """
    response = client.chat.completions.create(
        model=LLM,
        messages=messages,
        temperature=0.0,
        max_tokens=max_tokens,
        extra_body=NO_THINK,
        response_format=JUDGE_RESPONSE_FORMAT
    )
    value = json.loads(response.choices[0].message.content)

    score = value.get("score")
    reason = value.get("reason")
    if type(score) is not int or not 1 <= score <= 5:
        raise ValueError(f"Некорректная оценка судьи: {score!r}")
    if not isinstance(reason, str) or not reason.strip():
        raise ValueError("Судья не вернул обоснование")
    return score, reason.strip()


def build_context(found_context):
    """Собирает промпт: system-инструкция + нумерованный контекст + вопрос."""
    return "\n".join(f"[{n}] {retrieval[3]}" for n, retrieval in enumerate(found_context, 1))


def build_message(prompt: str) -> dict[str, str]:
    return [{"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": prompt}]


def judge_faithfulness(context, answer):
    """Оценивает верность ответа контексту."""
    messages =  build_message(FAITH_PROMPT.format(context=context, answer=answer))
    return generate_judge(messages)


def judge_relevance(question, answer):
    """Оценивает соответствие ответа вопросу."""
    messages =  build_message(REL_PROMPT.format(question=question, answer=answer))
    return generate_judge(messages)

In [55]:
with open("queries_with_answers.json", 'r') as f:
    queries = json.load(f)

In [56]:
judge_faith, judge_rel = [], []

judged_rows = []

for i, query in enumerate(queries):
    print(f"Номер: {i}; вопрос: {query['query'][:40]}")
    rel_id = query['relevant_docs']
    
    hits = retrieve(query["query"], col, e5_base_embedder, k=30)
    bm25_search = sparse_search(query["query"], k=30)
    hybrid = hybrid_search(hits, bm25_search, k=30)
    reranked_hits = rerank(query["query"], hybrid[:15], reranker)

    context = build_context(reranked_hits)

    faith, faith_reason = judge_faithfulness(context, query["predict_answer"])
    rel, rel_reason = judge_relevance(query["query"], query["predict_answer"])

    judge_faith.append(faith)
    judge_rel.append(rel)

    judged_rows.append(
        {
            "id": i,
            "faith": faith,
            "rel": rel,
            "faith_reason": faith_reason,
            "rel_reason": rel_reason,
        }
    )

    queries[i]["metrics"] = {
            "faith": faith,
            "rel": rel,
            "faith_reason": faith_reason,
            "rel_reason": rel_reason,
        }


def decision_agreement(scores, human_scores, threshold):
    """Считает совпадения решений по заданному порогу."""
    return sum(
        (score < threshold) == (human < threshold)
        for score, human in zip(scores, human_scores)
    )

print("Средний faithfulness:", round(float(np.mean(judge_faith)), 1))
print("Средний answer relevance:", round(float(np.mean(judge_rel)), 1))

print("Проблемные ответы по рабочему порогу:")
for row in judged_rows:
    if row["faith"] < 4 or row["rel"] < 3:
        print(row)

Номер: 0; вопрос: что означает ошибка P-102
Номер: 1; вопрос: какая версия webOS или Tizen нужна для п
Номер: 2; вопрос: обязателен ли HDCP 2.2 для просмотра в 4
Номер: 3; вопрос: как подключить саундбар по HDMI ARC
Номер: 4; вопрос: что означает ошибка P-109
Номер: 5; вопрос: что значит ошибка P-111
Номер: 6; вопрос: сколько стоит тариф Максимум и что в нег
Номер: 7; вопрос: сколько дней длится пробный период
Номер: 8; вопрос: какая скорость интернета нужна для 4K
Номер: 9; вопрос: есть ли приложение Фильмоскопа для Ninte
Номер: 10; вопрос: сколько профилей можно создать в одном а
Номер: 11; вопрос: на каком тарифе доступен звук 5.1
Номер: 12; вопрос: сколько устройств можно привязать к акка
Номер: 13; вопрос: сколько загрузок можно хранить на одном 
Номер: 14; вопрос: поддерживается ли телевизор QX-450
Номер: 15; вопрос: вечером фильмы постоянно тормозят и крут
Номер: 16; вопрос: включаю кино — на экране всё идёт, а из 
Номер: 17; вопрос: телевизор показывает чёрный экран, но зв
Номе

In [57]:
with open("queries_with_metrics.json", "w") as f:
    json.dump(queries, f, ensure_ascii=False)

In [58]:
for i, row in enumerate(judged_rows):
    if row["faith"] < 4 or row["rel"] < 3:
        print(f"\nНайденные чанки проблемного запроса {row['id']}:")
        try:
            
            retrieval = build_context(all_reranked[row['id']])
            print(retrieval)
        except:
            print("gap")
        print("\nАктуальные чанки:")

        rel_ids = queries[i]["relevant_docs"]
        for rel_id in rel_ids:
            print(docs[rel_id]['cleared_text'])

        print(f"faithfulness = {row['faith']}, reason = {row['faith_reason']}")
        print(f"relevance answer = {row['rel']}, reason = {row['rel_reason']}")


Найденные чанки проблемного запроса 6:
[1] Тарифы «Фильмоскопа» Тариф выбирается при оформлении подписки и в любой момент виден в настройках аккаунта, в разделе «Подписка»: там указаны название текущего тарифа, дата следующего списания и способ оплаты. Там же тариф меняется на другой — порядок перехода описан в отдельном разделе справки. От тарифа зависит только то, как вы смотрите: сколько экранов работает одновременно, до какого качества поднимается картинка и доступно ли скачивание на мобильные устройства. Число привязанных к аккаунту устройств и число профилей от тарифа не зависят — эти лимиты одинаковы для всех подписок. Сервис работает по подписке. Доступны три тарифа: «Мобильный» — 199 ₽ в месяц. Просмотр на одном устройстве, только телефон или планшет. Качество до 720p. «Стандарт» — 399 ₽ в месяц. Просмотр на двух устройствах одновременно, качество до Full HD (1080p), доступно скачивание на мобильные устройства. «Максимум» — 599 ₽ в месяц. Просмотр на четырёх устройствах однов

6 запрос: видим, что чанк найден был релевантный, однако модель сгаллюционировала, отсюда плохой faithfullness и relevance_answer;<br>

27 запрос: faithfullness = 2, видимо из-за нерелевантных далее чанков, модель дала не совсем корректный ответ;<br>

33 запрос: faithfullness = 2, актуальный был найден, но он не топ-1 и скорее всего модель не увидела его, так как он был в середине, а модель плохо видит контекст в середине, relevance_answer = 3 - часть вопросов была закрыта;<br>

44: faithfullness = 3, главная часть закрыта, но была придумана какая-то часть видимо из-за нерелевантных чанков, relevance_answer = 3 - часть вопросов также закрыта;<br>

52: faithfullness = 3 - часть выдумана, relevance_answer = 3 - часть вопрос закрыта; <br>

53: faithfullness = 2 - часть выдумана, relevance_answer = 5 - все вопросы закрыты; <br>

55: faithfullness = 5 - все идеально, relevance_answer = 1 - модель сгаллюционировала; <br>

57, 59, 60, 62, 64: faithulness = 5, так как релевантынх чанков тут нет и модель на это хорошо ответила; <br>

65: faithulness = 2 видимо модель сгенерировала ответ на нерелевантных чанках, relevance_answer = 5 - видимо чанки не нужны были, чтобы ответить корректно на этот вопрос



In [59]:
JUDGE_CORRECT = (
    "Ты — строгий эксперт. Сравни ОТВЕТ с ЭТАЛОННЫМ ФАКТОМ. Верни 1, если ответ "
    "передаёт эталонный факт (формулировка может отличаться), и 0, если ответ "
    "противоречит эталону, неполон или отказывается отвечать. "
    "Верни только одну цифру: 1 или 0.")

In [65]:
for i, query in enumerate(queries):
    question = query["query"]
    print(f"№ запроса: {i}, запрос: {question}")
    gold = query["gold"]
    answer = query["predict_answer"]
    correct = generate(
        messages =[
            {"role": "system", "content": JUDGE_CORRECT},
            {"role": "user", "content": 
             f"ВОПРОС: {question}\nЭТАЛОННЫЙ ФАКТ:{gold}\n\n"
             f"ОТВЕТ: {answer}"
            }
            ]
    )

    queries[i]["metrics"]["correctness"] = correct


№ запроса: 0, запрос: что означает ошибка P-102
№ запроса: 1, запрос: какая версия webOS или Tizen нужна для приложения на телевизоре
№ запроса: 2, запрос: обязателен ли HDCP 2.2 для просмотра в 4K
№ запроса: 3, запрос: как подключить саундбар по HDMI ARC
№ запроса: 4, запрос: что означает ошибка P-109
№ запроса: 5, запрос: что значит ошибка P-111
№ запроса: 6, запрос: сколько стоит тариф Максимум и что в него входит
№ запроса: 7, запрос: сколько дней длится пробный период
№ запроса: 8, запрос: какая скорость интернета нужна для 4K
№ запроса: 9, запрос: есть ли приложение Фильмоскопа для Nintendo Switch
№ запроса: 10, запрос: сколько профилей можно создать в одном аккаунте
№ запроса: 11, запрос: на каком тарифе доступен звук 5.1
№ запроса: 12, запрос: сколько устройств можно привязать к аккаунту
№ запроса: 13, запрос: сколько загрузок можно хранить на одном устройстве
№ запроса: 14, запрос: поддерживается ли телевизор QX-450
№ запроса: 15, запрос: вечером фильмы постоянно тормозят и кр

In [85]:
import pandas as pd
queries_pd = []
METRICS = ['faithfullness', 'answer_relevance', "correctness"]

for query in queries:
    queries_pd.append(query['metrics'])

queries_pd = pd.DataFrame(queries_pd)
queries_pd = queries_pd[["faith", "rel", "correctness"]]

queries_pd = queries_pd.rename(columns={
    "faith": "faithfullness",
    "rel": "answer_relevance"
})

queries_pd["correctness"] = queries_pd['correctness'].astype("int")

report = queries_pd[METRICS].mean().round(3)

In [91]:
precision, recall, mrr = calculate_metrics(queries_filter, [qid for qid in range(len(queries_filter))], all_reranked)

report_retrieval = pd.Series(
    {
        "precision@5": round(precision, 3),
        "recall@5": round(recall, 3),
        "mrr": round(mrr, 3)
    }
)

индекс    запрос                                       |R|    P@5    R@5     RR   CATS
0         что означает ошибка P-102                      1   0.20   1.00   1.00   {'Ошибки воспроизведения'}
1         какая версия webOS или Tizen нужна для при     1   0.20   1.00   1.00   {'Устройства и качество'}
2         обязателен ли HDCP 2.2 для просмотра в 4K      1   0.20   1.00   1.00   {'Подписка и оплата', 'Устройства и качество'}
3         как подключить саундбар по HDMI ARC            1   0.20   1.00   1.00   {'Ошибки воспроизведения', 'Устройства и качество'}
4         что означает ошибка P-109                      1   0.20   1.00   1.00   {'Ошибки воспроизведения'}
5         что значит ошибка P-111                        1   0.20   1.00   1.00   {'Ошибки воспроизведения'}
6         сколько стоит тариф Максимум и что в него      1   0.20   1.00   1.00   {'Функции и контент', 'Подписка и оплата', 'Устройства и качество'}
7         сколько дней длится пробный период             1   0.20

In [93]:
report = pd.concat([report, report_retrieval])

In [94]:
report

faithfullness       4.712
answer_relevance    4.515
correctness         0.924
precision@5         0.250
recall@5            0.955
mrr                 0.829
dtype: float64

# Итоговый вердикт по готовности системы к пилоту

## 1. Достигнуты ли целевые ориентиры

**Да, все три целевых ориентира достигнуты на тестовом наборе.**

| Ориентир | Порог | Факт | Статус |
|---|---|---|---|
| `recall@5` (итоговый поиск, по запросам с ответом в базе) | ≥ 0,85 | **0,955** | ✅ достигнут с запасом |
| `correctness` (доля ответов с эталонным фактом, весь набор) | ≥ 0,90 | **0,924** | ✅ достигнут (запас ~2,4 п.п.) |
| Отказ на вопросах без ответа в базе (без выдумывания фактов, согласованная фраза) | качественный критерий | **подтверждён**: модель не выдумывает факты и возвращает согласованную фразу отказа | ✅ достигнут |

**Оговорки:**

- **`correctness` 0,924** — запас над порогом небольшой (~2,4 п.п.). На пилоте с более шумными реальными запросами метрика может просесть ниже 0,90 — нужен мониторинг.
- **`precision@5` = 0,250** — низкая, но это *не* целевой ориентир. Объяснима стратегией «широкого» набора кандидатов: высокий `recall@5` (0,955) ценой `precision`. Для RAG это допустимо, если генератор устойчив к шуму в контексте — что подтверждают `faithfulness` 4,712 и `correctness` 0,924.

| Метрика | Значение | Комментарий |
|---|---|---|
| `faithfulness` | 4,712 | высокое качество, факты подтверждены контекстом |
| `answer_relevance` | 4,515 | высокое соответствие ответа вопросу |
| `correctness` | 0,924 | ✅ порог ≥ 0,90, запас ~2,4 п.п. |
| `precision@5` | 0,250 | ⚠️ низкая, но не целевой ориентир |
| `recall@5` | 0,955 | ✅ порог ≥ 0,85, запас солидный |
| `mrr` | 0,829 | эталонный факт в среднем высоко в выдаче |

## 2. Готова ли система к пилоту

**Да, готова к пилоту.**

Все три целевых ориентира пройдены, включая поведение на вопросах без ответа в базе. Качество генерации высокое (`faithfulness` и `answer_relevance` > 4,5 из 5). Единственное условие запуска — настроить мониторинг метрик на пилотном трафике, так как запас `correctness` над порогом тонкий.

### Рекомендация 1. Мониторинг `correctness` и `recall@5` на пилотном трафике

**Что сделать:** настроить алерты при падении ниже порогов (`correctness` < 0,90, `recall@5` < 0,85), а также отслеживать долю корректных отказов на out-of-scope запросах.

**Зачем:** запас `correctness` всего ~2,4 п.п., реальные запросы шумнее тестового набора. Раннее обнаружение деградации позволит откатиться или дообучить модель до того, как пользователи увидят ошибки. Поведение отказа подтверждено на тесте — важно убедиться, что оно сохраняется на реальном трафике.

### Рекомендация 2. Разбор кейсов с низким `precision@5`

**Что сделать:** проверить, не теряется ли эталонный факт в шуме контекста; при необходимости добавить реранкер или сузить top-k для генерации.

**Зачем:** `precision@5` = 0,250 означает много нерелевантного в контексте (в нашем случае в среднем был 1 релевантный документ, а передавали в модель топ-5). Сейчас генератор справляется (`correctness` 0,924), но на сложных реальных запросах шум может снизить `correctness` и `faithfulness`. Реранкер повысит устойчивость без потери recall.